In [ ]:
# Installations
# !conda install requests spacy dwdsmor sfst pyndl pygam ftfy sfst langid
# !conda install spacy de_zdl_lg --extra-index-url https://gitup.uni-potsdam.de/api/v4/projects/21461/packages/pypi/simple # de_zdl_lg is the best model for German morphology (GUM/ZDL corpus)
# !sudo apt install -y python3 default-jdk libsaxonhe-java sfst

In [14]:
import io
import os
import re
import tarfile
from itertools import islice
from urllib.parse import unquote

import pandas as pd
import numpy as np
import requests
import spacy
import langid
import sfst
import dwdsmor
# import dwdsmor.spacy
from charset_normalizer import from_bytes
from ftfy import fix_text
from tqdm import tqdm

In [15]:
import dwdsmor

# Initialize the generator
paradigm_gen = dwdsmor.paradigm()

# Generate the full inflectional table for a noun
# Example: 'Haus'
forms = paradigm_gen('Haus')

for form in forms:
    # Filter for the Nominative Plural form
    if form.pos == 'N' and form.case == 'Nom' and form.number == 'Pl':
        print(f"Lemma: {form.lemma}, Plural: {form.wordform}")

AttributeError: module 'dwdsmor' has no attribute 'paradigm'

# download and combine corpus files

In [ ]:
#webtexts between 2011 and 2023
URL_LIST = url_list = [
  #2011 2018 2021
#     "https://downloads.wortschatz-leipzig.de/corpora/deu_web_2011_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-com_web_2018_1M.tar.gz",
    "https://downloads.wortschatz-leipzig.de/corpora/deu-com_web_2021_1M.tar.gz",
cont
#   #AT 2012 2014 2015 2019 2023
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-at_web_2023_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-at_web_2019_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-at_web_2015_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-at_web_2014_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-at_web_2012_1M.tar.gz",

#   #DE 2020 2021
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-de_web_2021_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-de_web_2020_1M.tar.gz",

#   #LU 2019 2021
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-lu_web_2021_1M.tar.gz",
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-lu_web_2019_1M.tar.gz",

#   #LI 2019
#     "https://downloads.wortschatz-leipzig.de/corpora/deu-li_web_2019_1M.tar.gz",
]

In [3]:
def build_corpus(url_list, output_file):
    """
    Merge corpora into one file

    url_list - list of tar.gz URLs to merge (list)
    output_file - how you want the output to be called including the file extension (string)

    Returns: output file
    """
    #download data from urls
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for url in tqdm(url_list, desc="Processing urls"):
            try:
                response = requests.get(url, stream=True)
                if response.status_code != 200:
                    print(f"Failed to download {url}")
                    continue
                
                #find relevant sentence file in downloaded folders
                with tarfile.open(fileobj=io.BytesIO(response.content), mode='r:gz') as tar_files: #r:gz=open for reading with gzip compression
                    sentences_file = next((m for m in tar_files if m.name.endswith("-sentences.txt")), None) #return the next member of the archive
                    if not sentences_file:
                        continue

                    #process sentences
                    f = tar_files.extractfile(sentences_file)
                    raw_data = f.read()

                    #encoding detection
                    detection = from_bytes(raw_data).best()
                    encoding_guess = detection.encoding
                    content = raw_data.decode(encoding_guess, errors='replace') #use 'replace' so the script doesn't crash on one bad byte

                    for line in content.splitlines():
                        text_parts = line.strip().split('\t', 1) #format: <ID>\t<SENTENCE>, so split at most at 1 tab

                        if len(text_parts) >= 2:
                            sentence = text_parts[1] #ID = text[0]
                            if sentence and len(sentence) > 5: # Filter out short junk
                                outfile.write(sentence + '\n')
            except Exception as e:
                print(f"Error processing {url}: {e}")

    print(f"\nCorpus saved and cleaned: {output_file}")

build_corpus(URL_LIST, "leipzig_deweb21_raw.txt")

Processing urls: 100%|██████████| 1/1 [00:16<00:00, 16.41s/it]


Corpus saved and cleaned: leipzig_deweb21_raw.txt


# clean corpus

- which POS tagger, spacy small or dwds which is german morphology specific?

to clean:
- corrupted chars/encoding from unicode/urls
    1. ftfy on each line
- embedded metadata
    - regex for predictable patterns: Source Tags: r"\" / Timestamps: r"@ \d{2}:\d{2}:\d{2}.*"

- special chars, emoticons, list symbols
    1. Use unquote from urllib.parse to fix %0A (newlines) and %20 (spaces). 
    2. Use a regex that removes leading non-alphanumeric characters
    3. remove punctuation 
        - Regex: re.sub(r"^[^\w\s]+", "", line) (removes symbols at the start of the line).
- english, russian, dutch
    - use langid to check lang
- bibliography text
- boilerplates
    1. save a set of lines to remove duplicates
    2. Keywords: Filter out lines containing "MwSt," "Versandkosten," "Alle Rechte vorbehalten," or "Impressum."
- truncated lines
    - Length Filter: Discard any line with fewer than 5 words or 30 characters.

In [13]:
CORPUS_FILE = "leipzig_deweb21_raw.txt"
OUTPUT_FILE = "leipzig_deweb21_clean.csv"
LIMIT_LINES = 10 #change to test on smaller amount of lines

nlp = spacy.load("de_core_news_lg", disable=["ner", "parser"]) #POS tagger

def count_lines(file_path):
    """Count lines for the progress bar total."""
    with open(file_path, 'rb') as f:
        return sum(1 for _ in f)

def clean_line(line):
    """
    Cleans, normalises, and validates a raw input string.    
    Decodes encoding artifacts, standardises whitespace, and applies heuristic filters (minimum length, digit density, boilerplate, German languages). 
    
    Returns the cleaned string or None if the line is discarded.
    """
    #mojibake
    line = unquote(line).strip()
    line = fix_text(line)
    
    #remove broken encoding
    if not line or "\ufffd" in line or "½" in line or "�" in line:
        return None

    #remove multiple spaces
    line = re.sub(r'(\b[a-zA-ZäöüßÄÖÜ])\s+(?=[a-zA-ZäöüßÄÖÜ]{2,})', r'\1', line)

    #space and dash normalisation
    line = line.replace('%20', ' ').replace('%0A', ' ')
    line = re.sub(r"[\s\u00A0\u1680\u180E\u2000-\u200B\u202F\u205F\u3000]+", " ", line)
    line = re.sub(r"[\u2010\u2011\u2012\u2013\u2014\u2015]", "-", line)

    #sentence fragment filter
    if not line.endswith(('.', '!', '?')):
        return None

    #remove metadata and noise
    line = re.sub(r"@ \d{2}:\d{2}:\d{2}.*", "", line)
    line = re.sub(r"\\", "", line)
    line = re.sub(r"^\d+[\s.]+", "", line) 
    line = re.sub(r"^[^\w\s]+", "", line).strip()

    #remove high density of digits
    if not line: #prevent division by 0
        return None
    if (sum(c.isdigit() for c in line) / len(line)) > 0.10:
        return None

    #remove repetitive boilerplate text / legalese
    bad_words = ["MwSt", "Versandkosten", "Impressum", "Copyright", "AGB", "Uhr", "Abschnitt"]
    if any(word in line for word in bad_words):
        return None

    #remove short junk
    if len(line) < 40 or len(line.split()) < 6:
        return None

    #language check
    lang, _ = langid.classify(line)
    if lang != 'de':
        return None

    return line

def get_structural_hash(line):
    """Remove structurally similar/identical sentences"""
    struct = re.sub(r'[^a-zäöüß]', '', line.lower())
    return hash(struct[:30])

def final_clean(text):
    """xxxx"""
    #empty lines
    if not text:
        return None
    #punctuation strip
    text = re.sub(r"[^a-zA-ZäöüßÄÖÜ\d\s-]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def process_corpus(limit=None):
    """xxx"""
    seen_sentences = set()
    structural_hashes = set()
    cleaned_sentences = []

    #get total lines for tqdm if limit is None
    total_to_process = limit if limit else count_lines(CORPUS_FILE)

    with open(CORPUS_FILE, "r", encoding="utf-8") as f:
        source = islice(f, limit) if limit else f
        for raw_line in tqdm(source, total=total_to_process, desc="Filtering"):
            parts = raw_line.strip().split('\t', 1)
            text = parts[1] if len(parts) > 1 else parts[0]
            
            cleaned = clean_line(text)
            if cleaned:
                s_hash = get_structural_hash(cleaned)
                if cleaned not in seen_sentences and s_hash not in structural_hashes:
                    cleaned_sentences.append(cleaned)
                    seen_sentences.add(cleaned)
                    structural_hashes.add(s_hash)

    data = []
    for doc in tqdm(nlp.pipe(cleaned_sentences, batch_size=50), 
                    total=len(cleaned_sentences), 
                    desc="Analysis"):
        for tok in doc:
            if tok.pos_ == "NOUN" and tok.morph.get("Number") == ["Plur"] and len(tok.text) >= 3:
            #and t.text.isalpha() #removes anything with digits in
                clean_sentence = final_clean(doc.text)
                clean_plural = final_clean(tok.text)   
                sg_lemma = tok.lemma_ 
                # Check for lemmatization artifacts
                # If lemma is longer than plural or very short, it's often a SpaCy error
                if len(sg_lemma) > len(clean_plural) or len(sg_lemma) < 3:
                    continue       
                data.append({"sentence": clean_sentence, "plurals": clean_plural, "lemma": sg_lemma})

    df = pd.DataFrame(data) 
    df.dropna(subset=['plurals']).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    return df

if __name__ == "__main__":
    results_df = process_corpus(limit=LIMIT_LINES)
    print(f"Saved {len(results_df)} valid sentences.")
#half an hour for 250k 
results_df.head(20)

OSError: [E050] Can't find model 'de_core_news_lg'. It doesn't seem to be a Python package or a valid path to a data directory.

In [12]:
results_df.head(20)

,sentence,plurals,lemma
0,100 Rabatt auf ein Lernen Zertifizierung Kurs ...,Rabatt,Rabatt
1,100 Virgin Resin Shrink Wrap mit maximalen UV-...,UV-Hemmern,UV-Hemmer
2,10-Kandidaten müssen es nicht sein aber sie so...,10-Kandidaten,"1,0-Kandidate"
3,10-Kandidaten müssen es nicht sein aber sie so...,Noten,Note
4,10 Seiten gepunktete Rasterseiten für Notizen ...,Seiten,Seite
5,10 Seiten gepunktete Rasterseiten für Notizen ...,Rasterseiten,Rasterseit
6,10 Seiten gepunktete Rasterseiten für Notizen ...,Notizen,Notiz
7,10 Seiten gepunktete Rasterseiten für Notizen ...,Ecken,Ecke
8,119000 mussten für Mein Kölner Dom Wrapped von...,mussten,mussten


In [ ]:
# Conceptual structure for processing DWDS-style lexical data
def process_dwds_lexicon(lexicon_file):
    data = []
    # Logic to parse dictionary entries rather than raw sentences
    for entry in lexicon_file:
        singular = entry['lemma']
        plural = entry['plural_form']
        
        # Categorize using the refined classification logic
        label = map_plural_to_class_robust({'plural_word': plural, 'lemma': singular})
        
        if label:
            data.append({
                "lemma": singular,
                "plural": plural,
                "label": label
            })
    return pd.DataFrame(data)

In [ ]:
import pandas as pd
import spacy
from tqdm.auto import tqdm  # Use .auto for best compatibility with scripts/notebooks

# 1. Load Data
df = pd.read_csv("leipzig_deweb21_clean.csv")

# 2. Handle multiple plurals
df['plural_word'] = df['plurals'].str.split(', ')
df = df.explode('plural_word')
df['plural_word'] = df['plural_word'].str.strip()

# 3. Lemmatization (Bottleneck 1)
nlp = spacy.load("de_core_news_sm")
sentences = df['sentence'].astype(str).tolist()
target_words = df['plural_word'].astype(str).tolist()
lemmas = []

# tqdm is correctly placed here to wrap the generator
# Added a 'desc' to clarify what is being processed


# 4. Classification Logic (Bottleneck 2)
def map_plural_to_class(row):
    p = str(row['plural_word']).lower().strip()
    s = str(row['lemma']).lower().strip()
    if s == p: return "0"
    if p.endswith('n'): return "n"
    if p.endswith('s'): return "s"
    if p.endswith('r'): return "r"
    if p.endswith('e'): return "e"
    return None

# Register tqdm with pandas to enable .progress_apply()
tqdm.pandas(desc="Classifying Plurals")

# Replace .apply() with .progress_apply()
df['class_label'] = df.progress_apply(map_plural_to_class, axis=1)

# 5. Final Cleanup
df = df.dropna(subset=['class_label'])
label_list = ["0", "n", "s", "e", "r"]
label2id = {l: i for i, l in enumerate(label_list)}
df['label_id'] = df['class_label'].map(label2id)

df.to_csv('deweb21_data.csv', index=False)

Lemmatizing:   0%|          | 0/1165645 [00:00<?, ?it/s]

Classifying Plurals:   0%|          | 0/1165645 [00:00<?, ?it/s]

In [4]:
df = pd.read_csv("deweb21_data.csv")
df.head(20)

,sentence,plurals,plural_word,lemma,class_label,label_id
0,100 Rabatt auf ein Lernen Zertifizierung Kurs ...,Rabatt,Rabatt,Rabatt,0,0
1,100 Virgin Resin Shrink Wrap mit maximalen UV-...,UV-Hemmern,UV-Hemmern,UV-Hemmer,n,1
2,10-Kandidaten müssen es nicht sein aber sie so...,"10-Kandidaten, Noten",10-Kandidaten,10-Kandidaten,0,0
3,10-Kandidaten müssen es nicht sein aber sie so...,"10-Kandidaten, Noten",Noten,Note,n,1
4,10 Seiten gepunktete Rasterseiten für Notizen ...,"Seiten, Rasterseiten, Notizen, Ecken",Seiten,Seite,n,1
5,10 Seiten gepunktete Rasterseiten für Notizen ...,"Seiten, Rasterseiten, Notizen, Ecken",Rasterseiten,Rasterseit,n,1
6,10 Seiten gepunktete Rasterseiten für Notizen ...,"Seiten, Rasterseiten, Notizen, Ecken",Notizen,Notiz,n,1
7,10 Seiten gepunktete Rasterseiten für Notizen ...,"Seiten, Rasterseiten, Notizen, Ecken",Ecken,Ecke,n,1
8,119000 mussten für Mein Kölner Dom Wrapped von...,mussten,mussten,mussn,n,1
9,16 der Bewerber über das eigene Xing Profil un...,Bewerber,Bewerber,Bewerber,0,0


# wug

In [2]:
plural_stimuli = {
    # --- Real Words (Baseline for Lexical Knowledge) ---
    "Park": {"sentence": "Es gibt viele [MASK] in Berlin.", "gender": "der"},
    "Biskuit": {"sentence": "Ich habe gestern zwanzig [MASK] gebacken.", "gender": "der"},
    "Jodel": {"sentence": "Max postet mindestens fünf [MASK] jeden Tag.", "gender": "der"},
    "Schal": {"sentence": "Monika hat viele [MASK] in ihrem Schrank.", "gender": "der"},
    "Test": {"sentence": "Schüler müssen heutzutage so viele [MASK] schreiben.", "gender": "der"},
    "Boykott": {"sentence": "Die meisten [MASK] sind politisch motiviert.", "gender": "der"},
    "Kabarett": {"sentence": "[MASK] sind ein Teil des Alltags, wenn man auf einem Kreuzfahrtschiff arbeitet.", "gender": "das"},
    "Bukett": {"sentence": "\"Wie viele [MASK] muss ich dir geben, bis du mir vergibst?\", fragte Klaus.", "gender": "das"},
    "Labor": {"sentence": "Da ihr Arbeitsplatz so viele [MASK] hat, hat Laura ganz vergessen, wo genau sie ihr Experiment verlassen hat.", "gender": "das"},
    "Fazit": {"sentence": "Wie viele [MASK] kann man aus einem Text ziehen?!", "gender": "das"},
    "Limit": {"sentence": "Es gibt keine [MASK]!", "gender": "das"},
    "Konto": {"sentence": "Wie viele [MASK] darf man haben?", "gender": "das"},
    "Pastorale": {"sentence": "Diese zwei [MASK] sind einfach wunderschön.", "gender": "die"},
    "Datscha": {"sentence": "Es gibt viele [MASK] auf dem Land in Russland.", "gender": "die"},
    "Playlist": {"sentence": "Die Vanessa hat so viele [MASK] auf Spotify!", "gender": "die"},
    "Tafel": {"sentence": "Das Schulbudget ist nicht groß genug, um für jedes Klassenzimmer neue [MASK] zu kaufen.", "gender": "die"},
    "Lawine": {"sentence": "[MASK] kommen im Winter oft vor.", "gender": "die"},
    "Wartezeit": {"sentence": "Die [MASK] beim Arzt sind oft echt lange.", "gender": "die"},
    "Teppich": {"sentence": "Unser Kunde braucht mindestens zehn [MASK].", "gender": "der"},
    "Hase": {"sentence": "Auf dem Rasen waren nur [MASK] zu sehen.", "gender": "der"},
    "Untertitel": {"sentence": "Die [MASK] in diesem Film sind einfach furbachtbar!", "gender": "der"},
    "Freizeichen": {"sentence": "[MASK] sind in jedem Land anders.", "gender": "das"},
    "Becken": {"sentence": "Unser Schwimmbad hat viele [MASK].", "gender": "das"},
    "Kissen": {"sentence": "Es ist immer voll bequem bei dem Daniel zu chillen, er hat immer so viele [MASK] auf seiner Couch.", "gender": "das"},

    # --- Nonce Words (Wug Test Items for Generalization) ---
    "Gupel": {"sentence": "Annika hat ihre zwei [MASK] geputzt.", "gender": "die"},
    "Maffer": {"sentence": "Wann habt ihr zum letzten Mal zwei [MASK] gesehen?", "gender": "die"},
    "Morfett": {"sentence": "Ich habe euch schon erzählt, dass die [MASK] schon zum Einsatz bereit sind.", "gender": "die"},
    "Trilit": {"sentence": "Lukas hat viele [MASK] bei sich Zuhause, wenn du eine brauchst.", "gender": "die"},
    "Knidoka": {"sentence": "Wir haben gestern über [MASK] gelesen.", "gender": "die"},
    "Grimzo": {"sentence": "Hast du die drei [MASK] gefunden?", "gender": "die"},
    "Trichel": {"sentence": "Gert und Birgit dachten sie hatten genug [MASK] fürs Abendessen gekauft.", "gender": "der"},
    "Pflechter": {"sentence": "Wir haben Flo gestern im Fernsehen gesehen, als seine Werbung für [MASK] gelaufen ist!", "gender": "der"},
    "Cherpak": {"sentence": "Mehrere [MASK] habe ich aber nie zusammen gesehen.", "gender": "der"},
    "Kniempe": {"sentence": "Kathi war richtig froh, ihre [MASK] wieder zurück zu haben.", "gender": "der"},
    "Pommeng": {"sentence": "Lena hat die [MASK] weggemacht.", "gender": "der"},
    "Sampa": {"sentence": "Ich kann es nicht leiden, wenn Kevin seine [MASK] auf dem Tisch stehen lässt!", "gender": "der"},
    "Fliffel": {"sentence": "Wo kann man gute [MASK] finden?", "gender": "das"},
    "Deipfer": {"sentence": "Hast du Julia schon die [MASK] gegeben?", "gender": "das"},
    "Istrat": {"sentence": "[MASK] sind meiner Meinung nach echt eklig.", "gender": "das"},
    "Virong": {"sentence": "Meine Mama sagt, dass [MASK] gut für die Gesundheit sind.", "gender": "das"},
    "Nofte": {"sentence": "Jedes Haus soll ein paar gute [MASK] haben!", "gender": "das"},
    "Brolck": {"sentence": "Sophie verlässt das Haus nie ohne [MASK].", "gender": "das"}
}

In [6]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertForMaskedLM

# 1. Setup
model_name = "deepset/gbert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)
model.eval()

# 2. Schema Setup
suffix_map = {"n": "##n", "s": "##s", "e": "##e", "r": "##er", "0": "."}
suffix_ids = {k: tokenizer.convert_tokens_to_ids(v) for k, v in suffix_map.items()}

def get_complete_analysis(lemma, sentence_template):
    # Prepare morphological probe
    probing_sentence = sentence_template.replace("[MASK]", f"{lemma}[MASK]")
    lemma_tokens = tokenizer.tokenize(lemma)
    
    # Inference
    inputs = tokenizer(probing_sentence, return_tensors="pt")
    mask_idx = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    
    with torch.no_grad():
        logits = model(**inputs).logits[0, mask_idx, :].squeeze()
    
    # Extract schema probabilities
    ending_logits = torch.stack([logits[suffix_ids[k]] for k in ["n", "s", "e", "r", "0"]])
    probs = torch.nn.functional.softmax(ending_logits, dim=-1).numpy()
    
    # Calculate Shannon Entropy: H = -sum(p * log2(p))
    entropy_val = -np.sum(probs * np.log2(probs + 1e-9))
    
    schemas = ["n", "s", "e", "r", "0"]
    winner = schemas[np.argmax(probs)]
    
    return lemma_tokens, winner, entropy_val

# 3. Process the DataFrame
rows = []
for lemma, info in plural_stimuli.items():
    tokens, pred, entropy = get_complete_analysis(lemma, info["sentence"])
    rows.append({
        "lemma": lemma,
        "gender": info["gender"],
        "lemma_tokens": tokens,
        "gbert_plural_schema": pred,
        "gbert_entropy": round(entropy, 4)
    })

gbert_df = pd.DataFrame(rows)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: deepset/gbert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Numpy is not available

In [4]:
gbert_df

,lemma,sentence,gender,gbert_plural_schema,lemma_tokens,schema_probs
0,Park,Es gibt viele [MASK] in Berlin.,der,r,[Park],"{'n': 0.004298332612961531, 's': 0.27079558372..."
1,Biskuit,Ich habe gestern zwanzig [MASK] gebacken.,der,e,"[Bis, ##ku, ##it]","{'n': 0.004579213447868824, 's': 0.40198013186..."
2,Jodel,Max postet mindestens fünf [MASK] jeden Tag.,der,s,"[Jo, ##del]","{'n': 0.018325461074709892, 's': 0.67940860986..."
3,Schal,Monika hat viele [MASK] in ihrem Schrank.,der,s,[Schal],"{'n': 0.00039243008359335363, 's': 0.879222273..."
4,Test,Schüler müssen heutzutage so viele [MASK] schr...,der,e,[Test],"{'n': 0.00036245808587409556, 's': 0.007600491..."
5,Boykott,Die meisten [MASK] sind politisch motiviert.,der,e,"[Boy, ##ko, ##tt]","{'n': 0.00024439452681690454, 's': 0.322743743..."
6,Kabarett,"[MASK] sind ein Teil des Alltags, wenn man auf...",das,s,"[Kabar, ##ett]","{'n': 0.024378839880228043, 's': 0.53607088327..."
7,Bukett,"""Wie viele [MASK] muss ich dir geben, bis du m...",das,s,"[Buk, ##ett]","{'n': 0.058574456721544266, 's': 0.49615377187..."
8,Labor,"Da ihr Arbeitsplatz so viele [MASK] hat, hat L...",das,e,[Labor],"{'n': 0.000630231574177742, 's': 0.17205084860..."
9,Fazit,Wie viele [MASK] kann man aus einem Text ziehen?!,das,s,[Fazit],"{'n': 0.009667589329183102, 's': 0.76592689752..."


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: deepset/gbert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Numpy is not available